# Step 5. The numbers behind the three charts

Everything the page draws comes from the measures of steps 2 to 4: their money, words and category, and their place on
the map. This notebook adds them up the way each chart does, saves the tables, and writes the one file the page reads.

**Money.** A measure's net cost over the cost table's five or six years. A measure that costs money is **new
spending**; one with a net saving or revenue (a spending review, a tax change that raises money) goes to **savings and
money raised**, whatever its category. A measure without a row in the cost tables counts for its words only.

**Gravity** (one budget at a time). Each category's share of the budget's new spending. The budget is **focused**
when its largest category has 35% or more of the new spending, **leaning** at 25% or more, **broad** otherwise.
Categories with 5% or more keep their colour; the rest, and Other, are grey. The second mass is what pays for it:
savings and money raised, by category. A budget is weighed only when at least 40% of its measures have a cost.

**Priorities** (one budget at a time, the 14 categories other than Other). Across: the category's share of new
spending, averaged over the last three budgets (each budget counts equally, whatever its size). Up: the category's
share of this budget's words. The lines sit at 1 in 14 (about 7%, an even share): Main priority (above on both),
Emerging (more words than money), Fading (more money than words), Low priority (below on both). Savings and money
raised are one more point: across, savings per $1 of new spending (three-budget average); up, the words of the
measures that save or raise money; "talked up" when those words are 7% or more of the budget's words.

**Map.** Each measure's place from step 4, coloured by its category.

**Budgets 2022 to 2026.** The page shows these five. Budget 2021 is read only for the money side of Priorities: the
three-budget averages of 2022 (2021 and 2022) and 2023 (2021 to 2023). It is in `category_year.csv` for that reason,
and nowhere else.

Outputs: `data/category_year.csv`, `data/gravity.csv`, `data/priorities.csv`, and `data/galaxy.json`, copied to `site/galaxy.json` (what the
page reads).

In [1]:
import json

import numpy as np
import pandas as pd

DATA = "data"
CUT_FOCUSED, CUT_LEANING = 0.35, 0.25      # Gravity: the largest category's share that makes a budget focused / leaning
COLOUR_MIN = 0.05                          # Gravity: categories with this share of new spending or more keep their colour
COSTED_MIN = 0.4                           # a budget is weighed when at least this share of its measures has a cost
WINDOW = 3                                 # Priorities: money averaged over the last three budgets
NEAR_LINE = 0.01                           # Priorities: within 1 point of a line is "near the line"
OTHER, SAVINGS = "Other", "Savings and money raised"
SHOW = [2022, 2023, 2024, 2025, 2026]      # the budgets on the page (Budget 2021: only in the Priorities averages)

cats = pd.read_csv(f"{DATA}/category_list.csv")                    # the 15 categories in their fixed order
NAMES = cats["category"].tolist()
QUAD = [c for c in NAMES if c != OTHER]                            # the 14 categories placed on Priorities
FAIR = 1 / len(QUAD)                                               # an even share: 1 in 14
print(f"{len(NAMES)} categories; Priorities places {len(QUAD)}; the even share is {FAIR:.1%}")

A = pd.read_csv(f"{DATA}/categorised.csv")                          # every categorised measure, Budget 2021 included
A["spend"] = A["money_b"].clip(lower=0).fillna(0)                  # new spending ($ billions)
A["save"] = (-A["money_b"]).clip(lower=0).fillna(0)                # savings and money raised ($ billions, positive)
A["saving"] = A["money_b"] < 0                                     # a measure that saves or raises money
ALL_YEARS = sorted(A["year"].unique())                             # for the base table and the Priorities averages
# the measures on the page: Budgets 2022 to 2026, with their place on the map (step 4)
M = A[A["year"].isin(SHOW)].merge(pd.read_csv(f"{DATA}/language.csv").drop(columns=["year", "heading", "category"]),
                                  on="id", how="left", validate="one_to_one")
assert M["x"].notna().all(), "a measure on the page has no place on the map: run step 4 again"
YEARS = SHOW
print(f"{len(A):,} measures read (Budgets {ALL_YEARS[0]} to {ALL_YEARS[-1]}); "
      f"on the page: {len(M):,} measures, Budgets {YEARS[0]} to {YEARS[-1]}")

15 categories; Priorities places 14; the even share is 7.1%
1,317 measures read (Budgets 2021 to 2026); on the page: 999 measures, Budgets 2022 to 2026


## By budget and category
The base table (Budget 2021 included, for the Priorities averages): for each budget and category, the number of measures (and how many have a cost), new spending,
savings and money raised, words (all, and those of the measures that save or raise money), and the shares used below.

In [2]:
rows = []
for year in ALL_YEARS:
    b = A[A["year"] == year]
    for name in NAMES:
        c = b[b["category"] == name]
        rows.append({"year": year, "category": name, "measures": len(c), "costed": int(c["money_b"].notna().sum()),
                     "new_spending_b": c["spend"].sum(), "savings_b": c["save"].sum(), "net_b": c["money_b"].sum(),
                     "words": int(c["words"].sum()), "words_saving": int(c.loc[c["saving"], "words"].sum())})
CY = pd.DataFrame(rows)
tot = CY.groupby("year")[["new_spending_b", "savings_b", "words"]].transform("sum")
CY["spending_share"] = CY["new_spending_b"] / tot["new_spending_b"]
CY["savings_share"] = CY["savings_b"] / tot["savings_b"]
CY["word_share"] = (CY["words"] - CY["words_saving"]) / tot["words"]   # words of its spending measures, of all words
CY.round(4).to_csv(f"{DATA}/category_year.csv", index=False)
pd.set_option("display.width", 200)
print("Share of new spending\n", CY.pivot(index="category", columns="year", values="spending_share")
      .reindex(NAMES).mul(100).round(1).to_string())

Share of new spending
 year                               2021  2022  2023  2024  2025  2026
category                                                             
Housing                             2.0  13.8   0.1  11.5   7.4  10.0
Affordability & social support     27.7   1.2   3.5  10.5  22.3  12.1
Health care                         5.4  12.7  42.6   4.3   0.0   1.0
Trade, industry & competitiveness   7.0   4.8   1.0   4.7   9.8  35.6
Jobs, skills & immigration         24.9   8.5   4.7   8.6   5.0   0.5
Infrastructure                      6.6   2.2   1.4   2.9   6.3   3.5
Research, AI & technology           2.0   3.1   0.7  10.6   1.1   2.1
Culture & communities               2.1   1.8   2.1   1.4   1.0   2.5
Nature & environment                2.7   4.1   1.3   0.9   0.8   1.2
Clean energy & climate              5.8  15.7  25.3   4.3   0.1   7.8
Taxation                            0.4   0.0   1.1   1.8   0.4   1.2
Government                          1.4   0.2   3.6   2.3   1.3   0

## Gravity: what each budget is about
Per budget: new spending, savings and money raised, the largest category and its share, the label, and the
*effective number of categories* (1 divided by the sum of squared shares: 1 if all the money went to one category,
14 if it were spread evenly) as a second measure of concentration.

In [3]:
G = []
for year in YEARS:
    c = CY[CY["year"] == year].set_index("category")
    s = c["spending_share"]
    top = s.idxmax()
    G.append({"year": year, "measures": int(c["measures"].sum()), "costed": int(c["costed"].sum()),
              "weighed": c["costed"].sum() >= COSTED_MIN * c["measures"].sum(),
              "new_spending_b": round(c["new_spending_b"].sum(), 3), "savings_b": round(c["savings_b"].sum(), 3),
              "top_category": top, "top_share": round(s[top], 4),
              "label": "focused" if s[top] >= CUT_FOCUSED else "leaning" if s[top] >= CUT_LEANING else "broad",
              "effective_categories": round(1 / (s ** 2).sum(), 2),
              "in_colour": ", ".join(f"{k} {v:.0%}" for k, v in s.sort_values(ascending=False).items()
                                     if v >= COLOUR_MIN and k != OTHER),
              "top_saving": c["savings_share"].idxmax() if c["savings_b"].sum() > 0 else ""})
G = pd.DataFrame(G)
G.to_csv(f"{DATA}/gravity.csv", index=False)
print(G.drop(columns=["in_colour"]).to_string(index=False))
print()
for _, g in G.iterrows():
    print(f"Budget {g['year']}: {g['label']}; in colour: {g['in_colour']}")

 year  measures  costed  weighed  new_spending_b  savings_b                      top_category  top_share   label  effective_categories                     top_saving
 2022       203     159     True          64.496     26.987                        Indigenous     0.1647   broad                  8.38                       Taxation
 2023       209     161     True          73.658     27.832                       Health care     0.4262 focused                  3.85                       Taxation
 2024       292     227     True          59.702     24.888                Defence & security     0.1991   broad                  8.80                       Taxation
 2025       118      86     True         143.842     54.162                Defence & security     0.4300 focused                  3.89                     Government
 2026       177     141     True         147.658      5.641 Trade, industry & competitiveness     0.3563 focused                  4.96 Affordability & social support

Bud

## Priorities: main, emerging, fading and low priority
For each budget on the page: its window (the last three budgets weighed), then for each category its money share in this budget,
its three-budget average (across), its share of this budget's words (up) and where that puts it.

In [4]:
# the averages may reach back to Budget 2021: weighed when at least 40% of its measures have a cost, as in Gravity
costed = CY.groupby("year")[["costed", "measures"]].sum()
WEIGHED = [int(u) for u in ALL_YEARS if costed.loc[u, "costed"] >= COSTED_MIN * costed.loc[u, "measures"]]


def shares(year):
    # money: each category's new spending, and savings, over the budget's new spending; words: over all its words
    c = CY[CY["year"] == year].set_index("category").reindex(NAMES)
    spend = c["new_spending_b"].sum()
    mon = list(c["new_spending_b"] / spend) + [c["savings_b"].sum() / spend]
    wd = list((c["words"] - c["words_saving"]) / c["words"].sum()) + [c["words_saving"].sum() / c["words"].sum()]
    return np.array(mon), np.array(wd)


P, PJ = [], {}
for year in [u for u in WEIGHED if u in SHOW]:
    win = [u for u in WEIGHED if year - WINDOW < u <= year]          # the last three budgets (fewer at the start)
    x = np.mean([shares(u)[0] for u in win], axis=0)                 # across: money share, three-budget average
    now, y = shares(year)                                            # this budget's money share, and its word share
    quad, near = [], []
    for k, name in enumerate(NAMES + [SAVINGS]):
        if name == SAVINGS:
            q = "talked up" if y[k] >= FAIR else "kept quiet"
        elif name == OTHER:
            q = ""                                                   # not placed
        else:
            q = ("Main" if y[k] >= FAIR else "Fading") if x[k] >= FAIR else ("Emerging" if y[k] >= FAIR else "Low priority")
        quad.append(q)
        near.append(bool(name not in (SAVINGS, OTHER) and (abs(x[k] - FAIR) < NEAR_LINE or abs(y[k] - FAIR) < NEAR_LINE)))
        P.append({"year": year, "window": f"{win[0]}–{win[-1]}" if len(win) > 1 else str(win[0]), "point": name,
                  "money_share_this_budget": round(now[k], 4), "money_share_window": round(x[k], 4),
                  "word_share": round(y[k], 4), "quadrant": q, "near_line": near[-1]})
    PJ[str(year)] = {"win": win, "x": [round(v, 5) for v in x], "y": [round(v, 5) for v in y],
                     "now": [round(v, 5) for v in now], "quad": quad, "near": near}
P = pd.DataFrame(P)
P.to_csv(f"{DATA}/priorities.csv", index=False)
for year, part in P.groupby("year"):
    print(f"\nBudget {year} (money: Budgets {part['window'].iloc[0]})")
    for q in ["Main", "Emerging", "Fading", "Low priority", "talked up", "kept quiet"]:
        names = part.loc[part["quadrant"] == q, "point"].tolist()
        if names:
            print(f"   {q:<13} {', '.join(names)}")


Budget 2022 (money: Budgets 2021–2022)
   Main          Housing, Jobs, skills & immigration, Clean energy & climate, Defence & security, Indigenous
   Emerging      Trade, industry & competitiveness
   Fading        Affordability & social support, Health care
   Low priority  Infrastructure, Research, AI & technology, Culture & communities, Nature & environment, Taxation, Government
   kept quiet    Savings and money raised

Budget 2023 (money: Budgets 2021–2023)
   Main          Affordability & social support, Health care, Jobs, skills & immigration, Clean energy & climate, Indigenous
   Emerging      Trade, industry & competitiveness, Defence & security
   Low priority  Housing, Infrastructure, Research, AI & technology, Culture & communities, Nature & environment, Taxation, Government
   kept quiet    Savings and money raised

Budget 2024 (money: Budgets 2022–2024)
   Main          Housing, Health care, Clean energy & climate, Indigenous
   Emerging      Affordability & social supp

## The page's file
`data/galaxy.json`: the budgets, the 15 categories (with their anchors on the map), every measure (budget, heading,
section, money, category, words, place on the map, novelty, closest earlier measures, the category it is worded like
and the policy it reverses, if any), and the Priorities numbers above, so the page draws exactly these tables. The
same file goes to `site/galaxy.json`, beside the page.

In [5]:
anchors = pd.read_csv(f"{DATA}/category_anchors.csv").set_index("category")
index = {i: k for k, i in enumerate(M["id"])}                       # measure id -> position in the file
cat_id = {c: k for k, c in enumerate(NAMES)}


def num(v, nd):
    return None if pd.isna(v) else round(float(v), nd)


measures = []
for _, m in M.iterrows():
    near = [[index[m[f"near{k}"]], float(m[f"near{k}_sim"])] for k in (1, 2, 3) if isinstance(m[f"near{k}"], str)]
    row = {"year": int(m["year"]), "title": m["heading"], "section": m["section"] if isinstance(m["section"], str) else "",
           "money": num(m["money_b"], 3), "cat": cat_id[m["category"]],
           "like": cat_id[m["like"]] if isinstance(m["like"], str) else None, "near": near,
           "novelty": num(m["novelty"], 2), "x": float(m["x"]), "y": float(m["y"]), "words": int(m["words"])}
    if isinstance(m["reverses"], str):
        row["reverses"] = m["reverses"]
    measures.append(row)

out = {"years": [int(y) for y in YEARS],
       "categories": [{"id": k, "name": c, "short": cats.loc[k, "short"], "n": int(anchors.loc[c, "measures"]),
                       "ax": float(anchors.loc[c, "ax"]), "ay": float(anchors.loc[c, "ay"])} for k, c in enumerate(NAMES)],
       "measures": measures,
       "priorities": PJ}
for path in (f"{DATA}/galaxy.json", "site/galaxy.json"):           # the data folder, and beside the page
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, separators=(",", ":"))
print(f"wrote {DATA}/galaxy.json and site/galaxy.json: {len(measures):,} measures, {len(NAMES)} categories, "
      f"Priorities for {len(PJ)} budgets")

wrote data/galaxy.json and site/galaxy.json: 999 measures, 15 categories, Priorities for 5 budgets
